<a href="https://colab.research.google.com/github/Guest558945/alexs/blob/main/IATradeCounterBlox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
from google import genai
from google.genai import types
import glob # ¡Nueva librería para leer múltiples archivos!
import os

# 1. Configuración de tu API
GOOGLE_API_KEY = ""
cliente = genai.Client(api_key=GOOGLE_API_KEY)

# 2. Leer TODOS los archivos de texto automáticamente
documento_precios = ""
# Busca todos los archivos que terminen en .txt en la carpeta principal
archivos_txt = glob.glob('/content/*.txt')

if len(archivos_txt) == 0:
    print("Error: No se encontraron archivos .txt. Asegúrate de subirlos a la carpeta /content/")
else:
    print(f"¡Se encontraron {len(archivos_txt)} archivos de listas! Cargándolos al cerebro del bot...")
    for ruta_archivo in archivos_txt:
        try:
            with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
                # Extraemos el nombre del archivo para mantener el orden internamente
                nombre_archivo = os.path.basename(ruta_archivo)
                documento_precios += f"\n\n--- INICIO DE DATOS: {nombre_archivo} ---\n"
                documento_precios += archivo.read()
        except Exception as e:
            print(f"Omitiendo un archivo por error de lectura: {e}")

# 3. Instrucciones
instrucciones_cb = f"""
Eres un experto trader y analista de mercado del juego 'Counter Blox' de Roblox.
Tu objetivo es evaluar trades (W, L o F), dar consejos de inversión y decir los precios exactos de las skins.

Aquí tienes la inmensa base de datos oficial y actualizada de precios y demandas del mercado (dividida en varios archivos):
{documento_precios}

Reglas:
- Basa tus precios SOLO en esta base de datos. Da Base Value, CK Value y Status.
- Responde con un tono gamer y amigable.
"""

# 4. Iniciar la sesión de chat en "memoria"
chat_gemini = cliente.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=instrucciones_cb,
        temperature=0.3
    )
)

# 5. La función "puente" entre la Interfaz Web y Gemini
def responder_chat(mensaje_usuario, historial_chat):
    try:
        respuesta = chat_gemini.send_message(mensaje_usuario)
        return respuesta.text
    except Exception as e:
        return f"Ups, hubo un error de conexión: {str(e)}"

# 6. ¡CONSTRUIR LA INTERFAZ WEB! (Manteniendo tu estilo oscuro)
interfaz = gr.ChatInterface(
    fn=responder_chat,
    title="🔪 CBTRADES IA",
    description="Tu analista personal del mercado de Counter Blox. Escribe una skin o un trade para evaluar.",
    theme="dark",
    examples=["¿Cuánto vale la AWP pinkVision?", "Me dan un Karambit Blood Widow por mi AK Bloodsport, ¿es buen trade?"]
)

# 7. Lanzar la página web
interfaz.launch(share=True, debug=True)

¡Se encontraron 34 archivos de listas! Cargándolos al cerebro del bot...


/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1143: UserWarning: Cannot load dark. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/dark' (Request ID: Root=1-69e98b45-3d718cb959d695242da39d31;352f1efc-e79b-4275-938b-523a4384e820)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://08b080e702b39866b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


AttributeError: module 'gradio' has no attribute 'blocks'